# Annotated Transformer — English → Arabic BPE edition

Updated from your notebook. Run cells top-to-bottom in a fresh Colab GPU runtime. Preserves the architecture explanations and attention visualizations. Replaces word-level processing with byte-level BPE, trains on up to 100k pairs, and saves self-contained best/last checkpoints.

**Before training:** review Part 3 settings (`TRAIN_LIMIT`, `MAX_LENGTH`, `ARTIFACT_DIR`, `RUN_MODE`). The copy-task training is optional. No trained model is bundled. Old cell outputs were cleared to avoid mixing baseline results with the new experiment.

**Resume:** set `RUN_MODE = "resume"` and retain the same data/model/batch/schedule settings; `num_epochs` is the desired total epoch count. Progress is saved after each completed epoch. Same-hardware RNG and loader state are restored, but bitwise reproducibility across hardware is not guaranteed.

**Limitations:** zero unknown input tokens does not guarantee translation quality. Filtered validation is a development subset, not the full official benchmark. BPE and word-level per-token losses are not directly comparable; compare decoded translations on a fixed set. Original source: https://nlp.seas.harvard.edu/annotated-transformer/


In [ ]:
# [IMPROVED] added "sacrebleu" | WHY: the standard, reproducible implementation of BLEU / chrF for the new test-set evaluation
%pip install -q "tokenizers>=0.20,<0.23" "altair>=5,<6" datasets pandas GPUtil "sacrebleu>=2.4,<3"


In [ ]:
import os
from os.path import exists
import math
import copy
import time
import warnings

import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

import pandas as pd
import altair as alt
import GPUtil

# Retained for compatibility with later notebook sections.
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

RUN_EXAMPLES = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
from IPython.display import display
from pathlib import Path
import hashlib
import json
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU is fine for the initial examples; use a GPU for training.")


PyTorch: 2.11.0+cu130
Device: cuda
GPU: Tesla T4


In [ ]:
def is_interactive_notebook():
    return __name__ == "__main__"


def show_example(fn, args=None):
    if is_interactive_notebook() and RUN_EXAMPLES:
        return fn(*(args if args is not None else []))


def execute_example(fn, args=None):
    if is_interactive_notebook() and RUN_EXAMPLES:
        fn(*(args if args is not None else []))


class DummyOptimizer:
    """Placeholder used in examples that do not update parameters."""

    def __init__(self):
        self.param_groups = [{"lr": 0.0}]

    def step(self):
        pass

    def zero_grad(self, set_to_none=False):
        pass


class DummyScheduler:
    """Placeholder used in examples without learning-rate scheduling."""

    def step(self):
        pass

> My comments are blockquoted. The main text is all from the paper itself.

# Background


The goal of reducing sequential computation also forms the
foundation of the Extended Neural GPU, ByteNet and ConvS2S, all of
which use convolutional neural networks as basic building block,
computing hidden representations in parallel for all input and
output positions. In these models, the number of operations required
to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and
logarithmically for ByteNet. This makes it more difficult to learn
dependencies between distant positions. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of
reduced effective resolution due to averaging attention-weighted
positions, an effect we counteract with Multi-Head Attention.

Self-attention, sometimes called intra-attention is an attention
mechanism relating different positions of a single sequence in order
to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading
comprehension, abstractive summarization, textual entailment and
learning task-independent sentence representations. End-to-end
memory networks are based on a recurrent attention mechanism instead
of sequencealigned recurrence and have been shown to perform well on
simple-language question answering and language modeling tasks.

To the best of our knowledge, however, the Transformer is the first
transduction model relying entirely on self-attention to compute
representations of its input and output without using sequence
aligned RNNs or convolution.

# Part 1: Model Architecture

# Model Architecture


Most competitive neural sequence transduction models have an
encoder-decoder structure
[(cite)](https://arxiv.org/abs/1409.0473). Here, the encoder maps an
input sequence of symbol representations $(x_1, ..., x_n)$ to a
sequence of continuous representations $\mathbf{z} = (z_1, ...,
z_n)$. Given $\mathbf{z}$, the decoder then generates an output
sequence $(y_1,...,y_m)$ of symbols one element at a time. At each
step the model is auto-regressive
[(cite)](https://arxiv.org/abs/1308.0850), consuming the previously
generated symbols as additional input when generating the next.

In [ ]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many
    other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [ ]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step."

    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)


The Transformer follows this overall architecture using stacked
self-attention and point-wise, fully connected layers for both the
encoder and decoder, shown in the left and right halves of Figure 1,
respectively.

![](https://github.com/harvardnlp/annotated-transformer/blob/master/images/ModalNet-21.png?raw=1)

> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.

## Encoder and Decoder Stacks

### Encoder

The encoder is composed of a stack of $N=6$ identical layers.


In [ ]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [ ]:
class Encoder(nn.Module):
    "Core encoder is a stack of N layers"

    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.


We employ a residual connection
[(cite)](https://arxiv.org/abs/1512.03385) around each of the two
sub-layers, followed by layer normalization
[(cite)](https://arxiv.org/abs/1607.06450).


In [ ]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.


That is, the output of each sub-layer is $\mathrm{LayerNorm}(x +
\mathrm{Sublayer}(x))$, where $\mathrm{Sublayer}(x)$ is the function
implemented by the sub-layer itself.  We apply dropout
[(cite)](http://jmlr.org/papers/v15/srivastava14a.html) to the
output of each sub-layer, before it is added to the sub-layer input
and normalized.

To facilitate these residual connections, all sub-layers in the
model, as well as the embedding layers, produce outputs of dimension
$d_{\text{model}}=512$.


In [ ]:
class SublayerConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """

    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        return x + self.dropout(sublayer(self.norm(x)))


Each layer has two sub-layers. The first is a multi-head
self-attention mechanism, and the second is a simple, position-wise
fully connected feed-forward network.

In [ ]:
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"

    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.

### Decoder

The decoder is also composed of a stack of $N=6$ identical layers.


In [ ]:
class Decoder(nn.Module):
    "Generic N layer decoder with masking."

    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


In addition to the two sub-layers in each encoder layer, the decoder
inserts a third sub-layer, which performs multi-head attention over
the output of the encoder stack.  Similar to the encoder, we employ
residual connections around each of the sub-layers, followed by
layer normalization.

In [ ]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"

    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)


We also modify the self-attention sub-layer in the decoder stack to
prevent positions from attending to subsequent positions.  This
masking, combined with fact that the output embeddings are offset by
one position, ensures that the predictions for position $i$ can
depend only on the known outputs at positions less than $i$.


> Below the attention mask shows the position each tgt word (row) is
> allowed to look at (column). Words are blocked for attending to
> future words during training.

In [ ]:
def subsequent_mask(size):
    """True = allowed attention; False = blocked attention."""
    return torch.ones(
        (1, size, size), dtype=torch.bool
    ).tril()


def example_mask():
    size = 20
    mask = subsequent_mask(size)[0]

    data = pd.DataFrame([
        {
            "Query position": query,
            "Key position": key,
            "Allowed": int(mask[query, key].item()),
        }
        for query in range(size)
        for key in range(size)
    ])

    return (
        alt.Chart(data)
        .mark_rect()
        .properties(
            height=300,
            width=300,
            title="Decoder causal mask: 1 = allowed, 0 = blocked",
        )
        .encode(
            x=alt.X("Key position:O", sort=list(range(size))),
            y=alt.Y("Query position:O", sort=list(range(size))),
            color=alt.Color(
                "Allowed:Q",
                scale=alt.Scale(domain=[0, 1], scheme="viridis"),
            ),
            tooltip=["Query position:O", "Key position:O", "Allowed:Q"],
        )
    )


show_example(example_mask)

alt.Chart(...)

### Attention

An attention function can be described as mapping a query and a set
of key-value pairs to an output, where the query, keys, values, and
output are all vectors.  The output is computed as a weighted sum of
the values, where the weight assigned to each value is computed by a
compatibility function of the query with the corresponding key.

We call our particular attention "Scaled Dot-Product Attention".
The input consists of queries and keys of dimension $d_k$, and
values of dimension $d_v$.  We compute the dot products of the query
with all keys, divide each by $\sqrt{d_k}$, and apply a softmax
function to obtain the weights on the values.



![](https://github.com/harvardnlp/annotated-transformer/blob/master/images/ModalNet-19.png?raw=1)


In practice, we compute the attention function on a set of queries
simultaneously, packed together into a matrix $Q$.  The keys and
values are also packed together into matrices $K$ and $V$.  We
compute the matrix of outputs as:

$$
   \mathrm{Attention}(Q, K, V) = \mathrm{softmax}(\frac{QK^T}{\sqrt{d_k}})V
$$

In [ ]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        # [IMPROVED] was: scores.masked_fill(mask == 0, -1e9)
        # WHY: -1e9 cannot be stored in float16 (its minimum is -65504): it becomes -inf and can give NaN with AMP.
        #      torch.finfo(dtype).min is the most negative value the current dtype can hold -> works in fp32 and fp16,
        #      and exp(min) is still exactly 0 after the softmax, so the masking result is the same.
        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn


The two most commonly used attention functions are additive
attention [(cite)](https://arxiv.org/abs/1409.0473), and dot-product
(multiplicative) attention.  Dot-product attention is identical to
our algorithm, except for the scaling factor of
$\frac{1}{\sqrt{d_k}}$. Additive attention computes the
compatibility function using a feed-forward network with a single
hidden layer.  While the two are similar in theoretical complexity,
dot-product attention is much faster and more space-efficient in
practice, since it can be implemented using highly optimized matrix
multiplication code.


While for small values of $d_k$ the two mechanisms perform
similarly, additive attention outperforms dot product attention
without scaling for larger values of $d_k$
[(cite)](https://arxiv.org/abs/1703.03906). We suspect that for
large values of $d_k$, the dot products grow large in magnitude,
pushing the softmax function into regions where it has extremely
small gradients (To illustrate why the dot products get large,
assume that the components of $q$ and $k$ are independent random
variables with mean $0$ and variance $1$.  Then their dot product,
$q \cdot k = \sum_{i=1}^{d_k} q_ik_i$, has mean $0$ and variance
$d_k$.). To counteract this effect, we scale the dot products by
$\frac{1}{\sqrt{d_k}}$.



![](https://github.com/harvardnlp/annotated-transformer/blob/master/images/ModalNet-20.png?raw=1)


Multi-head attention allows the model to jointly attend to
information from different representation subspaces at different
positions. With a single attention head, averaging inhibits this.

$$
\mathrm{MultiHead}(Q, K, V) =
    \mathrm{Concat}(\mathrm{head_1}, ..., \mathrm{head_h})W^O \\
    \text{where}~\mathrm{head_i} = \mathrm{Attention}(QW^Q_i, KW^K_i, VW^V_i)
$$

Where the projections are parameter matrices $W^Q_i \in
\mathbb{R}^{d_{\text{model}} \times d_k}$, $W^K_i \in
\mathbb{R}^{d_{\text{model}} \times d_k}$, $W^V_i \in
\mathbb{R}^{d_{\text{model}} \times d_v}$ and $W^O \in
\mathbb{R}^{hd_v \times d_{\text{model}}}$.

In this work we employ $h=8$ parallel attention layers, or
heads. For each of these we use $d_k=d_v=d_{\text{model}}/h=64$. Due
to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full
dimensionality.

In [ ]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        "Take in model size and number of heads."
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        # We assume d_v always equals d_k
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        "Implements Figure 2"
        if mask is not None:
            # Same mask applied to all h heads.
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)

        # 1) Do all the linear projections in batch from d_model => h x d_k
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]

        # 2) Apply attention on all the projected vectors in batch.
        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout
        )

        # 3) "Concat" using a view and apply a final linear.
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        return self.linears[-1](x)

### Applications of Attention in our Model

The Transformer uses multi-head attention in three different ways:
1) In "encoder-decoder attention" layers, the queries come from the
previous decoder layer, and the memory keys and values come from the
output of the encoder.  This allows every position in the decoder to
attend over all positions in the input sequence.  This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence
models such as [(cite)](https://arxiv.org/abs/1609.08144).


2) The encoder contains self-attention layers.  In a self-attention
layer all of the keys, values and queries come from the same place,
in this case, the output of the previous layer in the encoder.  Each
position in the encoder can attend to all positions in the previous
layer of the encoder.


3) Similarly, self-attention layers in the decoder allow each
position in the decoder to attend to all positions in the decoder up
to and including that position.  We need to prevent leftward
information flow in the decoder to preserve the auto-regressive
property.  We implement this inside of scaled dot-product attention
by masking out (setting to $-\infty$) all values in the input of the
softmax which correspond to illegal connections.

## Position-wise Feed-Forward Networks

In addition to attention sub-layers, each of the layers in our
encoder and decoder contains a fully connected feed-forward network,
which is applied to each position separately and identically.  This
consists of two linear transformations with a ReLU activation in
between.

$$\mathrm{FFN}(x)=\max(0, xW_1 + b_1) W_2 + b_2$$

While the linear transformations are the same across different
positions, they use different parameters from layer to
layer. Another way of describing this is as two convolutions with
kernel size 1.  The dimensionality of input and output is
$d_{\text{model}}=512$, and the inner-layer has dimensionality
$d_{ff}=2048$.

In [ ]:
class PositionwiseFeedForward(nn.Module):
    "Implements FFN equation."

    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.

## Embeddings and Softmax

Similarly to other sequence transduction models, we use learned
embeddings to convert the input tokens and output tokens to vectors
of dimension $d_{\text{model}}$.  We also use the usual learned
linear transformation and softmax function to convert the decoder
output to predicted next-token probabilities.  In our model, we
share the same weight matrix between the two embedding layers and
the pre-softmax linear transformation, similar to
[(cite)](https://arxiv.org/abs/1608.05859). In the embedding layers,
we multiply those weights by $\sqrt{d_{\text{model}}}$.


In [ ]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

## Positional Encoding

Since our model contains no recurrence and no convolution, in order
for the model to make use of the order of the sequence, we must
inject some information about the relative or absolute position of
the tokens in the sequence.  To this end, we add "positional
encodings" to the input embeddings at the bottoms of the encoder and
decoder stacks.  The positional encodings have the same dimension
$d_{\text{model}}$ as the embeddings, so that the two can be summed.
There are many choices of positional encodings, learned and fixed
[(cite)](https://arxiv.org/pdf/1705.03122.pdf).

In this work, we use sine and cosine functions of different frequencies:

$$PE_{(pos,2i)} = \sin(pos / 10000^{2i/d_{\text{model}}})$$

$$PE_{(pos,2i+1)} = \cos(pos / 10000^{2i/d_{\text{model}}})$$

where $pos$ is the position and $i$ is the dimension.  That is, each
dimension of the positional encoding corresponds to a sinusoid.  The
wavelengths form a geometric progression from $2\pi$ to $10000 \cdot
2\pi$.  We chose this function because we hypothesized it would
allow the model to easily learn to attend by relative positions,
since for any fixed offset $k$, $PE_{pos+k}$ can be represented as a
linear function of $PE_{pos}$.

In addition, we apply dropout to the sums of the embeddings and the
positional encodings in both the encoder and decoder stacks.  For
the base model, we use a rate of $P_{drop}=0.1$.



In [ ]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."

    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)


> Below the positional encoding will add in a sine wave based on
> position. The frequency and offset of the wave is different for
> each dimension.

In [ ]:
def example_positional():
    pe = PositionalEncoding(20, 0)
    y = pe.forward(torch.zeros(1, 100, 20))

    data = pd.concat(
        [
            pd.DataFrame(
                {
                    "embedding": y[0, :, dim],
                    "dimension": dim,
                    "position": list(range(100)),
                }
            )
            for dim in [4, 5, 6, 7]
        ]
    )

    return (
        alt.Chart(data)
        .mark_line()
        .properties(width=800)
        .encode(x="position", y="embedding", color="dimension:N")
        .interactive()
    )


show_example(example_positional)

alt.Chart(...)


We also experimented with using learned positional embeddings
[(cite)](https://arxiv.org/pdf/1705.03122.pdf) instead, and found
that the two versions produced nearly identical results.  We chose
the sinusoidal version because it may allow the model to extrapolate
to sequence lengths longer than the ones encountered during
training.

## Full Model

> Here we define a function from hyperparameters to a full model.

In [ ]:
def make_model(
    src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1,
    tie_weights=False,  # [IMPROVED] new option | default False = original model, so old checkpoints still load
):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model, dropout=dropout)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )

    # This was important from their code.
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    # [IMPROVED] weight tying: the target embedding E (V x d) and the output projection W (V x d) become ONE matrix.
    # WHY: done in the original paper (section 3.4). Saves V*d parameters (8000*256 = 2M for the improved model),
    #      and the shared matrix gets gradients from both the input side and the output side.
    if tie_weights:
        model.generator.proj.weight = model.tgt_embed[0].lut.weight
    return model


# [IMPROVED] new helper | WHY: report model size; parameters() counts a tied (shared) matrix only once
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

## Inference:

> Here we make a forward step to generate a prediction of the
model. We try to use our transformer to memorize the input. As you
will see the output is randomly generated due to the fact that the
model is not trained yet. In the next tutorial we will build the
training function and try to train our model to memorize the numbers
from 1 to 10.

In [ ]:
@torch.no_grad()
def inference_test():
    # Small, untrained model for checking the architecture.
    test_model = make_model(
        src_vocab=11,
        tgt_vocab=11,
        N=2,
        d_model=128,
        d_ff=512,
        h=4,
        dropout=0.0,
    )
    test_model.eval()

    src = torch.tensor(
        [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]],
        dtype=torch.long,
    )
    src_mask = torch.ones(1, 1, src.size(1), dtype=torch.bool)

    memory = test_model.encode(src, src_mask)

    # Token 0 is the start token for this synthetic example only.
    ys = torch.zeros(1, 1, dtype=torch.long)

    for _ in range(9):
        tgt_mask = subsequent_mask(ys.size(1))

        out = test_model.decode(memory, src_mask, ys, tgt_mask)

        # Predict from the last decoder position.
        log_probs = test_model.generator(out[:, -1])
        next_token = log_probs.argmax(dim=-1, keepdim=True)

        ys = torch.cat([ys, next_token], dim=1)

    assert ys.shape == (1, 10)
    print("Architecture check passed.")
    print("Untrained model prediction:", ys.tolist())


def run_tests():
    inference_test()


show_example(run_tests)

Architecture check passed.
Untrained model prediction: [[0, 7, 3, 7, 3, 7, 3, 9, 3, 9]]


# Part 2: Model Training

# Training

This section describes the training regime for our models.


> We stop for a quick interlude to introduce some of the tools
> needed to train a standard encoder decoder model. First we define a
> batch object that holds the src and target sentences for training,
> as well as constructing the masks.

## Batches and Masking

In [ ]:
class Batch:
    """Hold source/target tensors and their attention masks."""

    def __init__(self, src, tgt=None, pad=2):
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)

        if tgt is not None:
            # Decoder input excludes the final token.
            self.tgt = tgt[:, :-1]

            # Prediction targets exclude the start token.
            self.tgt_y = tgt[:, 1:]

            self.tgt_mask = self.make_std_mask(self.tgt, pad)
            self.ntokens = (self.tgt_y != pad).sum()

    @staticmethod
    def make_std_mask(tgt, pad):
        padding_mask = (tgt != pad).unsqueeze(-2)
        causal_mask = subsequent_mask(tgt.size(-1)).to(tgt.device)
        return padding_mask & causal_mask


> Next we create a generic training and scoring function to keep
> track of loss. We pass in a generic loss compute function that
> also handles parameter updates.

## Training Loop

In [ ]:
from dataclasses import dataclass


@dataclass
class TrainState:
    """Cumulative counters across calls when the state is reused."""

    step: int = 0
    accum_step: int = 0
    samples: int = 0
    tokens: int = 0
    skipped_steps: int = 0  # [IMPROVED] AMP updates skipped because gradients overflowed (inf/NaN)


def run_epoch(
    data_iter,
    model,
    loss_compute,
    optimizer,
    scheduler,
    mode="train",
    accum_iter=1,
    train_state=None,
    scaler=None,      # [IMPROVED] GradScaler for mixed precision (None = original behaviour)
    clip_norm=None,   # [IMPROVED] max gradient norm (None = no clipping = original behaviour)
    use_amp=False,    # [IMPROVED] float16 autocast forward pass, CUDA only (False = original behaviour)
    log_every=40,     # [IMPROVED] print interval was hard-coded to 40; now configurable (0 = silent)
):
    if accum_iter < 1:
        raise ValueError("accum_iter must be at least 1.")

    if train_state is None:
        train_state = TrainState()

    is_training = mode in ("train", "train+log")
    model.train(is_training)

    # [IMPROVED] AMP only helps on a GPU, so it is switched off automatically on CPU.
    device_type = next(model.parameters()).device.type
    use_amp = bool(use_amp) and device_type == "cuda"
    use_scaler = scaler is not None and scaler.is_enabled()

    total_loss = 0.0
    total_tokens = 0
    pending_batches = 0
    pending_tokens = 0
    start = time.time()

    if is_training:
        optimizer.zero_grad(set_to_none=True)

    def update_parameters(token_count):
        # [IMPROVED] with AMP the gradients are multiplied by the loss scale S; unscale_ divides them by S.
        # WHY: must happen BEFORE the division by token_count and BEFORE clipping, so both see the true gradients.
        if use_scaler:
            scaler.unscale_(optimizer)

        # Convert accumulated summed-loss gradients into
        # gradients of the mean loss per non-padding token.
        for group in optimizer.param_groups:
            for parameter in group["params"]:
                if parameter.grad is not None:
                    parameter.grad.div_(token_count)

        # [IMPROVED] gradient clipping: if ||g|| > clip_norm, rescale g to length clip_norm (direction unchanged).
        # WHY: protects the bigger model against rare very large updates.
        if clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        # [IMPROVED] was: optimizer.step()
        # WHY: scaler.step skips the update if any gradient is inf/NaN; scaler.update halves S after an
        #      overflow and doubles it after 2000 clean steps. S only drops when the step was skipped.
        if use_scaler:
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            stepped = scaler.get_scale() >= scale_before
        else:
            optimizer.step()
            stepped = True
        optimizer.zero_grad(set_to_none=True)

        # Advance the schedule once per parameter update.
        # [IMPROVED] only when the update really happened, so the LR schedule follows real updates.
        if stepped:
            scheduler.step()
            train_state.accum_step += 1
        else:
            train_state.skipped_steps += 1

    for i, batch in enumerate(data_iter):
        ntokens = int(batch.ntokens.item())

        if ntokens == 0:
            continue

        # Evaluation avoids building a gradient graph.
        with torch.set_grad_enabled(is_training):
            # [IMPROVED] autocast: matrix multiplications run in float16 on the GPU (Tensor Cores -> faster);
            # softmax / log_softmax / the loss stay in float32. enabled=False -> exactly the original code.
            with torch.autocast(device_type=device_type, dtype=torch.float16, enabled=use_amp):
                out = model(
                    batch.src,
                    batch.tgt,
                    batch.src_mask,
                    batch.tgt_mask,
                )

                loss, loss_node = loss_compute(
                    out, batch.tgt_y, batch.ntokens
                )

            if is_training:
                # loss_node is normalized by batch.ntokens.
                # Recover summed loss for token-weighted accumulation.
                # [IMPROVED] was: (loss_node * ntokens).backward()
                # WHY: with AMP the loss is multiplied by S first, so small float16 gradients do not underflow to 0.
                if use_scaler:
                    scaler.scale(loss_node * ntokens).backward()
                else:
                    (loss_node * ntokens).backward()

        loss_value = (
            loss.detach().item()
            if isinstance(loss, torch.Tensor)
            else float(loss)
        )

        total_loss += loss_value
        total_tokens += ntokens

        if is_training:
            train_state.step += 1
            train_state.samples += batch.src.size(0)
            train_state.tokens += ntokens

            pending_batches += 1
            pending_tokens += ntokens

            if pending_batches == accum_iter:
                update_parameters(pending_tokens)
                pending_batches = 0
                pending_tokens = 0

            if log_every and i % log_every == 0:  # [IMPROVED] was: if i % 40 == 0:
                elapsed = max(time.time() - start, 1e-9)
                lr = optimizer.param_groups[0]["lr"]

                print(
                    f"Batch: {i:5d} | "
                    f"Loss/token: {loss_value / ntokens:.4f} | "
                    f"Tokens/sec: {total_tokens / elapsed:.1f} | "
                    f"LR: {lr:.2e}"
                )

        del out, loss, loss_node

    # Apply gradients from any remaining incomplete accumulation group.
    if is_training and pending_batches > 0:
        update_parameters(pending_tokens)

    if total_tokens == 0:
        raise ValueError("No non-padding target tokens found in this epoch.")

    return total_loss / total_tokens, train_state

> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.

## Training Data and Batching

We trained on the standard WMT 2014 English-German dataset
consisting of about 4.5 million sentence pairs.  Sentences were
encoded using byte-pair encoding, which has a shared source-target
vocabulary of about 37000 tokens. For English-French, we used the
significantly larger WMT 2014 English-French dataset consisting of
36M sentences and split tokens into a 32000 word-piece vocabulary.


Sentence pairs were batched together by approximate sequence length.
Each training batch contained a set of sentence pairs containing
approximately 25000 source tokens and 25000 target tokens.


> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.

## Hardware and Schedule

We trained our models on one machine with 8 NVIDIA P100 GPUs.  For
our base models using the hyperparameters described throughout the
paper, each training step took about 0.4 seconds.  We trained the
base models for a total of 100,000 steps or 12 hours. For our big
models, step time was 1.0 seconds.  The big models were trained for
300,000 steps (3.5 days).


## Optimizer

We used the Adam optimizer [(cite)](https://arxiv.org/abs/1412.6980)
with $\beta_1=0.9$, $\beta_2=0.98$ and $\epsilon=10^{-9}$.  We
varied the learning rate over the course of training, according to
the formula:

$$
lrate = d_{\text{model}}^{-0.5} \cdot
  \min({step\_num}^{-0.5},
    {step\_num} \cdot {warmup\_steps}^{-1.5})
$$

This corresponds to increasing the learning rate linearly for the
first $warmup\_steps$ training steps, and decreasing it thereafter
proportionally to the inverse square root of the step number.  We
used $warmup\_steps=4000$.


> Note: This part is very important. Need to train with this setup
> of the model.


> Example of the curves of this model for different model sizes and
> for optimization hyperparameters.

In [ ]:
def rate(step, model_size, factor, warmup):
    step = max(step, 1)

    return factor * (
        model_size ** (-0.5)
        * min(step ** (-0.5), step * warmup ** (-1.5))
    )


def example_learning_schedule():
    configurations = [
        (512, 1, 4000),
        (512, 1, 8000),
        (256, 1, 4000),
    ]

    # Sample every 20 steps to keep the chart lightweight.
    steps = sorted(set([1] + list(range(20, 20001, 20))))

    rows = []
    for model_size, factor, warmup in configurations:
        for step in steps:
            rows.append({
                "Step": step,
                "Learning rate": rate(
                    step, model_size, factor, warmup
                ),
                "Configuration": f"d_model={model_size}, warmup={warmup}",
            })

    return (
        alt.Chart(pd.DataFrame(rows))
        .mark_line()
        .properties(width=650, height=300)
        .encode(
            x=alt.X("Step:Q", title="Optimizer update"),
            y=alt.Y("Learning rate:Q"),
            color="Configuration:N",
            tooltip=["Step:Q", "Learning rate:Q", "Configuration:N"],
        )
        .interactive()
    )


show_example(example_learning_schedule)

alt.Chart(...)

## Regularization

### Label Smoothing

During training, we employed label smoothing of value
$\epsilon_{ls}=0.1$ [(cite)](https://arxiv.org/abs/1512.00567).
This hurts perplexity, as the model learns to be more unsure, but
improves accuracy and BLEU score.


> We implement label smoothing using the KL div loss. Instead of
> using a one-hot target distribution, we create a distribution that
> has `confidence` of the correct word and the rest of the
> `smoothing` mass distributed throughout the vocabulary.


> Here we can see an example of how the mass is distributed to the
> words based on confidence.

In [ ]:
class LabelSmoothing(nn.Module):
    """KL-divergence loss with label smoothing and ignored padding."""

    def __init__(self, size, padding_idx, smoothing=0.0):
        super().__init__()

        if size <= 2:
            raise ValueError("Vocabulary size must be greater than 2.")
        if not 0.0 <= smoothing < 1.0:
            raise ValueError("smoothing must be in [0, 1).")
        if not 0 <= padding_idx < size:
            raise ValueError("padding_idx must be a valid vocabulary ID.")

        self.criterion = nn.KLDivLoss(reduction="sum")
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist = None

    def forward(self, x, target):
        # x: [number_of_tokens, vocabulary_size], log-probabilities
        # target: [number_of_tokens], correct token IDs
        assert x.size(1) == self.size

        with torch.no_grad():
            true_dist = torch.full_like(
                x, self.smoothing / (self.size - 2)
            )

            # Assign most probability mass to the correct token.
            true_dist.scatter_(
                1, target.unsqueeze(1), self.confidence
            )

            # Never assign target probability to the padding class.
            true_dist[:, self.padding_idx] = 0.0

            # Ignore positions whose correct target is padding.
            true_dist[target == self.padding_idx] = 0.0

        self.true_dist = true_dist
        return self.criterion(x, true_dist)


def example_label_smoothing():
    # Toy vocabulary: 5 tokens; padding ID = 0 for this example only.
    crit = LabelSmoothing(size=5, padding_idx=0, smoothing=0.4)

    # Finite log-probabilities, avoiding log(0).
    logits = torch.tensor([
        [-2.0, 0.0, 2.0, -1.0, -2.0]
    ]).repeat(5, 1)

    log_probs = logits.log_softmax(dim=-1)
    targets = torch.tensor([2, 1, 0, 3, 3], dtype=torch.long)

    loss = crit(log_probs, targets)

    distribution = crit.true_dist.cpu()
    rows = [
        {
            "Target probability": distribution[row, col].item(),
            "Vocabulary ID": col,
            "Example": row,
        }
        for row in range(5)
        for col in range(5)
    ]

    print(f"Summed KL-divergence loss: {loss.item():.4f}")

    return (
        alt.Chart(pd.DataFrame(rows))
        .mark_rect()
        .properties(width=300, height=250)
        .encode(
            x=alt.X("Vocabulary ID:O"),
            y=alt.Y("Example:O"),
            color=alt.Color(
                "Target probability:Q",
                scale=alt.Scale(domain=[0, 1], scheme="viridis"),
            ),
            tooltip=[
                "Example:O",
                "Vocabulary ID:O",
                "Target probability:Q",
            ],
        )
    )


show_example(example_label_smoothing)

Summed KL-divergence loss: 4.8847


alt.Chart(...)


> Label smoothing actually starts to penalize the model if it gets
> very confident about a given choice.

In [ ]:
def loss(x, crit):
    # Relative weights: padding, correct token, three other tokens.
    weights = torch.tensor(
        [[1e-9, float(x), 1.0, 1.0, 1.0]],
        dtype=torch.float32,
    )

    probabilities = weights / weights.sum(dim=-1, keepdim=True)
    target = torch.tensor([1], dtype=torch.long)

    with torch.no_grad():
        value = crit(probabilities.log(), target)

    return probabilities[0, 1].item(), value.item()


def penalization_visualization():
    crit = LabelSmoothing(size=5, padding_idx=0, smoothing=0.1)

    data = pd.DataFrame(
        [loss(x, crit) for x in range(1, 100)],
        columns=["Correct-token probability", "Loss"],
    )

    return (
        alt.Chart(data)
        .mark_line()
        .properties(
            width=450,
            height=250,
            title="Label smoothing penalizes overconfidence",
        )
        .encode(
            x=alt.X(
                "Correct-token probability:Q",
                axis=alt.Axis(format=".0%"),
            ),
            y=alt.Y("Loss:Q", title="KL-divergence loss"),
            tooltip=["Correct-token probability:Q", "Loss:Q"],
        )
        .interactive()
    )


show_example(penalization_visualization)

alt.Chart(...)

# A First  Example

> We can begin by trying out a simple copy-task. Given a random set
> of input symbols from a small vocabulary, the goal is to generate
> back those same symbols.

## Synthetic Data

In [ ]:
def data_gen(V, batch_size, nbatches, device="cpu"):
    """Synthetic copy task: padding=0, start=1, content tokens=2..V-1."""
    for _ in range(nbatches):
        data = torch.randint(
            2, V, size=(batch_size, 10), device=device
        )
        data[:, 0] = 1

        yield Batch(data.clone(), data.clone(), pad=0)

## Loss Computation

## Greedy Decoding

> This code predicts a translation using greedy decoding for simplicity.

In [ ]:
class SimpleLossCompute:
    """Return summed loss for reporting and normalized loss for training."""

    def __init__(self, generator, criterion):
        self.generator = generator
        self.criterion = criterion

    def __call__(self, x, y, norm):
        # [IMPROVED] added .float() | WHY: keep the loss in float32 even when the forward pass runs in float16 (AMP)
        log_probs = self.generator(x).float()

        summed_loss = self.criterion(
            log_probs.reshape(-1, log_probs.size(-1)),
            y.reshape(-1),
        )

        normalized_loss = summed_loss / norm

        # Matches our run_epoch interface.
        return summed_loss.detach(), normalized_loss


@torch.no_grad()
def greedy_decode(
    model, src, src_mask, max_len, start_symbol, end_symbol=None
):
    """Decode one source sentence; max_len includes the start token."""
    if src.size(0) != 1:
        raise ValueError("This decoder expects one source sentence.")

    was_training = model.training
    model.eval()

    try:
        memory = model.encode(src, src_mask)

        ys = torch.full(
            (1, 1),
            start_symbol,
            dtype=torch.long,
            device=src.device,
        )

        for _ in range(max_len - 1):
            tgt_mask = subsequent_mask(ys.size(1)).to(src.device)

            out = model.decode(
                memory, src_mask, ys, tgt_mask
            )
            log_probs = model.generator(out[:, -1])
            next_token = log_probs.argmax(dim=-1, keepdim=True)

            ys = torch.cat([ys, next_token], dim=1)

            if (
                end_symbol is not None
                and next_token.item() == end_symbol
            ):
                break

        return ys

    finally:
        model.train(was_training)

In [ ]:
# [IMPROVED] NEW CELL: batched greedy decoding + beam search
#
# Translation score: log P(y|x) = sum_t log P(y_t | y_<t, x)
# - Greedy picks the single best token at every step and can never go back.
# - Beam search keeps the k best partial translations ("beams"). Each step extends every beam by every
#   token (k*V candidates) and keeps the best k. A candidate ending in </s> moves to the finished list.
#   Stops when k translations are finished or the length limit is reached. With k = 1 it equals greedy.
# - Length penalty (GNMT, Wu et al. 2016): every extra token adds a negative log P, so raw scores prefer
#   short outputs. Finished candidates are compared by  log P(y|x) / lp(y),  lp(y) = ((5 + |y|) / 6) ** alpha,  alpha = 0.6
# - greedy_decode_batch: same result as greedy_decode, but many sentences at once (fast test-set evaluation).

@torch.no_grad()
def greedy_decode_batch(
    model, src, src_mask, max_len, start_symbol=0, end_symbol=1, pad_symbol=2
):
    """Greedy decoding for a padded batch of sources -> [B, <=max_len]."""
    was_training = model.training
    model.eval()
    try:
        memory = model.encode(src, src_mask)
        batch_size = src.size(0)
        ys = torch.full((batch_size, 1), start_symbol, dtype=torch.long, device=src.device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=src.device)
        for _ in range(max_len - 1):
            tgt_mask = subsequent_mask(ys.size(1)).to(src.device)
            out = model.decode(memory, src_mask, ys, tgt_mask)
            next_token = model.generator(out[:, -1]).argmax(dim=-1)
            # Sentences that already produced </s> only receive padding.
            next_token = next_token.masked_fill(finished, pad_symbol)
            ys = torch.cat([ys, next_token.unsqueeze(1)], dim=1)
            finished |= next_token == end_symbol
            if bool(finished.all()):
                break
        return ys
    finally:
        model.train(was_training)


@torch.no_grad()
def beam_search_decode(
    model, src, src_mask, max_len, start_symbol, end_symbol,
    beam_size=4, length_penalty=0.6,
):
    """Beam search for ONE source sentence; returns [1, T] like greedy_decode."""
    if src.size(0) != 1:
        raise ValueError("This decoder expects one source sentence.")

    def lp(length):
        # GNMT length normalization; length counts generated tokens incl. </s>.
        return ((5.0 + length) / 6.0) ** length_penalty

    was_training = model.training
    model.eval()
    try:
        device = src.device
        memory = model.encode(src, src_mask)
        ys = torch.full((1, 1), start_symbol, dtype=torch.long, device=device)
        scores = torch.zeros(1, device=device)  # summed log-probs of alive beams
        finished = []  # (normalized score, token list)

        for _ in range(max_len - 1):
            k = ys.size(0)
            out = model.decode(
                memory.expand(k, -1, -1), src_mask.expand(k, -1, -1),
                ys, subsequent_mask(ys.size(1)).to(device),
            )
            log_probs = model.generator(out[:, -1]).float()  # [k, V]
            vocab = log_probs.size(-1)
            # score(y_<t, v) = score(y_<t) + log P(v | y_<t, x), for all k*V candidates
            candidates = (scores.unsqueeze(1) + log_probs).view(-1)
            top_scores, top_ids = candidates.topk(min(2 * beam_size, candidates.numel()))

            new_rows, new_scores = [], []
            for score, flat_id in zip(top_scores.tolist(), top_ids.tolist()):
                beam, token = divmod(flat_id, vocab)
                sequence = ys[beam].tolist() + [token]
                if token == end_symbol:
                    finished.append((score / lp(len(sequence) - 1), sequence))
                else:
                    new_rows.append(sequence)
                    new_scores.append(score)
                if len(new_rows) == beam_size:
                    break

            if len(finished) >= beam_size or not new_rows:
                break
            ys = torch.tensor(new_rows, dtype=torch.long, device=device)
            scores = torch.tensor(new_scores, device=device)

        if not finished:  # length limit reached: rank the alive beams instead
            finished = [
                (score / lp(len(sequence) - 1), sequence)
                for score, sequence in zip(scores.tolist(), ys.tolist())
            ]
        best = max(finished, key=lambda item: item[0])[1]
        return torch.tensor([best], dtype=torch.long, device=device)
    finally:
        model.train(was_training)

In [ ]:
def example_simple_model():
    torch.manual_seed(42)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print("Training on:", device)

    V = 11

    criterion = LabelSmoothing(
        size=V, padding_idx=0, smoothing=0.0
    ).to(device)

    # Smaller model for this preliminary experiment.
    model = make_model(
        src_vocab=V,
        tgt_vocab=V,
        N=2,
        d_model=128,
        d_ff=512,
        h=4,
        dropout=0.1,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1.0,
        betas=(0.9, 0.98),
        eps=1e-9,
    )

    lr_scheduler = LambdaLR(
        optimizer,
        lr_lambda=lambda step: rate(
            step,
            model_size=128,
            factor=1.0,
            warmup=400,
        ),
    )

    loss_compute = SimpleLossCompute(model.generator, criterion)
    train_state = TrainState()
    batch_size = 80

    # Reuse the same validation examples across epochs.
    validation_batches = list(
        data_gen(V, batch_size, 5, device=device)
    )

    for epoch in range(20):
        train_loss, train_state = run_epoch(
            data_gen(V, batch_size, 20, device=device),
            model,
            loss_compute,
            optimizer,
            lr_scheduler,
            mode="train",
            train_state=train_state,
        )

        val_loss, _ = run_epoch(
            validation_batches,
            model,
            loss_compute,
            DummyOptimizer(),
            DummyScheduler(),
            mode="eval",
        )

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train loss: {train_loss:.4f} | "
            f"Validation loss: {val_loss:.4f}"
        )

    # Use the same start token and content range as training.
    src = torch.tensor(
        [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]],
        dtype=torch.long,
        device=device,
    )
    src_mask = (src != 0).unsqueeze(-2)

    prediction = greedy_decode(
        model,
        src,
        src_mask,
        max_len=src.size(1),
        start_symbol=1,
    )

    print("\nInput:     ", src[0].tolist())
    print("Prediction:", prediction[0].tolist())
    print("Exact match:", torch.equal(src, prediction))

    return model


# Optional synthetic preflight (not needed for translation training):
# copy_model = show_example(example_simple_model)


# Part 3: English → Arabic with byte-level BPE

This adaptation trains a small Transformer from scratch on OPUS-100. Separate English and Arabic BPE tokenizers learn from training text only, using all 256 byte symbols. Validation and test text never train the tokenizers. The test split is reserved for final evaluation.

Defaults: 100,000 candidate training pairs, 8,000 vocabulary entries per language, maximum 128 subword tokens including boundaries, 2 encoder/decoder layers, 128 hidden dimensions, 4 heads, 20 epochs. Length filtering can reduce the retained dataset. This is an educational baseline, not a pretrained translator.

Use a GPU runtime. Set `ARTIFACT_DIR` to a mounted persistent directory if you want checkpoints to survive a Colab reset. `RUN_MODE="resume"` resumes from the last completed epoch; `RUN_MODE="load"` loads the best model without training. Old word-level checkpoints are incompatible.


## Data and BPE tokenizers


In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from torch.nn.utils.rnn import pad_sequence

# [IMPROVED] experiment switch | WHY: run the original settings ("baseline") and the new settings ("improved")
#            with the same code, then compare both on the same test set with BLEU / chrF.
RUN_EXPERIMENT = "improved"  # "baseline" or "improved"

SEED = 42
# [IMPROVED] was: TRAIN_LIMIT = 100000
# WHY: the baseline validation loss was still falling at epoch 20 (3.666 -> 3.661) = underfitting,
#      so the bigger model gets 2x more training pairs. "baseline" keeps the original 100000.
TRAIN_LIMIT = 100000 if RUN_EXPERIMENT == "baseline" else 200000
VALID_LIMIT = 2000
MAX_LENGTH = 128
BPE_VOCAB_SIZE = 8000
SPECIALS = ["<s>", "</s>", "<blank>", "<unk>"]
# [IMPROVED] was: ARTIFACT_DIR = Path("en_ar_bpe_run")
# WHY: each experiment gets its own folder (the two runs never overwrite each other), stored in Google Drive
#      when possible so checkpoints survive a Colab reset (then RUN_MODE = "resume" continues training).
USE_DRIVE = True
BASE_DIR = Path("en_ar_runs")
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/en_ar_runs")
    except Exception as error:  # not on Colab, or mounting was declined
        print("Google Drive not mounted; saving locally instead:", error)
ARTIFACT_DIR = BASE_DIR / RUN_EXPERIMENT
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RUN_MODE = "train"  # train (fresh), resume (last epoch), or load (best)
BEST_MODEL_PATH = ARTIFACT_DIR / "best.pt"
LAST_MODEL_PATH = ARTIFACT_DIR / "last.pt"
torch.manual_seed(SEED)
raw_data = load_dataset("Helsinki-NLP/opus-100", "ar-en")


In [ ]:
def collect_text_pairs(split, limit):
    pairs, seen = [], set()
    for row in split.shuffle(seed=SEED):
        en, ar = (row["translation"][lang].strip() for lang in ("en", "ar"))
        if not en or not ar or max(len(en), len(ar)) > 2000:
            continue
        if (en, ar) in seen or any(t in en or t in ar for t in SPECIALS):
            continue
        seen.add((en, ar))
        pairs.append((en, ar))
        if len(pairs) >= limit:
            break
    return pairs

valid_texts = collect_text_pairs(raw_data["validation"], VALID_LIMIT)
train_texts = collect_text_pairs(raw_data["train"], TRAIN_LIMIT)
valid_sources = {en for en, _ in valid_texts}
train_texts = [(en, ar) for en, ar in train_texts if en not in valid_sources]
assert train_texts and valid_texts
print("Candidate train / validation pairs:", len(train_texts), len(valid_texts))


Candidate train / validation pairs: 100000 1991


In [ ]:
def train_bpe(texts):
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=BPE_VOCAB_SIZE, min_frequency=2,
        special_tokens=SPECIALS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer

if RUN_MODE in ("resume", "load"):
    restore_path = LAST_MODEL_PATH if RUN_MODE == "resume" else BEST_MODEL_PATH
    metadata = torch.load(restore_path, map_location="cpu", weights_only=True)
    tokenizer_en = Tokenizer.from_str(metadata["tokenizer_en"])
    tokenizer_ar = Tokenizer.from_str(metadata["tokenizer_ar"])
    MAX_LENGTH = metadata["max_length"]
    del metadata
elif RUN_MODE == "train":
    tokenizer_en = train_bpe(en for en, _ in train_texts)
    tokenizer_ar = train_bpe(ar for _, ar in train_texts)
else:
    raise ValueError("RUN_MODE must be train, resume, or load")

for tok in (tokenizer_en, tokenizer_ar):
    assert [tok.token_to_id(t) for t in SPECIALS] == [0, 1, 2, 3]

test_text = "أين الكنز الثمين ؟"
assert tokenizer_ar.decode(tokenizer_ar.encode(test_text).ids) == test_text
assert 3 not in tokenizer_ar.encode(test_text).ids
print("Arabic round-trip:", tokenizer_ar.decode(tokenizer_ar.encode(test_text).ids))


Arabic round-trip: أين الكنز الثمين ؟


In [ ]:
class Vocabulary:
    def __init__(self, tokens):
        self.itos = list(tokens)
        self.stoi = {
            token: index for index, token in enumerate(self.itos)
        }
        self.default_index = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def __call__(self, tokens):
        return [self[token] for token in tokens]

    def get_stoi(self):
        return self.stoi.copy()

    def get_itos(self):
        return self.itos.copy()

    def lookup_tokens(self, indices):
        return [self.itos[int(index)] for index in indices]

    def set_default_index(self, index):
        self.default_index = index



def vocabulary_from_tokenizer(tokenizer):
    return Vocabulary([tokenizer.id_to_token(i) for i in range(tokenizer.get_vocab_size())])

vocab_src = vocabulary_from_tokenizer(tokenizer_en)
vocab_tgt = vocabulary_from_tokenizer(tokenizer_ar)
print("English / Arabic vocabulary sizes:", len(vocab_src), len(vocab_tgt))


English / Arabic vocabulary sizes: 8000 8000


In [ ]:
def encode_bpe_pairs(text_pairs):
    encoded, kept = [], []
    for en, ar in text_pairs:
        s, t = tokenizer_en.encode(en).ids, tokenizer_ar.encode(ar).ids
        if max(len(s), len(t)) > MAX_LENGTH - 2:
            continue
        encoded.append((torch.tensor([0] + s + [1]), torch.tensor([0] + t + [1])))
        kept.append((en, ar))
    print(f"Kept {len(encoded):,}; length-filtered {len(text_pairs)-len(encoded):,}")
    return encoded, kept

train_encoded, train_texts_kept = encode_bpe_pairs(train_texts)
valid_encoded, valid_texts_kept = encode_bpe_pairs(valid_texts)
assert train_encoded and valid_encoded

def encoded_unknown_rate(pairs, index):
    total = sum(pair[index].numel() - 2 for pair in pairs)
    unknown = sum(int((pair[index][1:-1] == 3).sum()) for pair in pairs)
    return unknown / max(total, 1)

for name, pairs in [("Training", train_encoded), ("Validation", valid_encoded)]:
    print(name, "English unknown:", f"{encoded_unknown_rate(pairs, 0):.2%}",
          "Arabic unknown:", f"{encoded_unknown_rate(pairs, 1):.2%}")

def collate_batch(batch):
    sources, targets = zip(*batch)
    return (pad_sequence(sources, batch_first=True, padding_value=2),
            pad_sequence(targets, batch_first=True, padding_value=2))

# [IMPROVED] new class: length bucketing
# WHY: a batch is padded to its longest sentence; lengths 5, 6, 40 -> 3*40 = 120 positions for 51 real tokens
#      (57.5% wasted). Shuffle -> cut into big pools -> sort each pool by length -> slice into batches ->
#      shuffle the batches: similar lengths inside a batch (little padding), random order across batches.
#      It has its own random generator, which is saved in the checkpoint for resume.
class BucketBatchSampler(torch.utils.data.Sampler):
    def __init__(self, lengths, batch_size, pool_factor=100, seed=SEED):
        self.lengths = list(lengths)
        self.batch_size = batch_size
        self.pool_size = batch_size * pool_factor
        self.generator = torch.Generator().manual_seed(seed)

    def __iter__(self):
        order = torch.randperm(len(self.lengths), generator=self.generator).tolist()
        batches = []
        for start in range(0, len(order), self.pool_size):
            pool = sorted(order[start:start + self.pool_size],
                          key=lambda index: self.lengths[index])
            batches += [pool[j:j + self.batch_size]
                        for j in range(0, len(pool), self.batch_size)]
        for k in torch.randperm(len(batches), generator=self.generator).tolist():
            yield batches[k]

    def __len__(self):
        n, full = len(self.lengths), self.pool_size
        return (n // full) * math.ceil(full / self.batch_size) + math.ceil((n % full) / self.batch_size)


def example_length(pair):  # [IMPROVED] helper for bucketing
    return max(pair[0].numel(), pair[1].numel())


def create_dataloaders(batch_size=32, bucketing=False):  # [IMPROVED] new option bucketing (False = original loaders)
    if bucketing:
        # [IMPROVED] training batches from BucketBatchSampler; validation batches = length-sorted fixed batches
        # (the order does not change the validation loss, it only removes padding)
        sampler = BucketBatchSampler([example_length(p) for p in train_encoded], batch_size)
        train_loader = DataLoader(train_encoded, batch_sampler=sampler,
                                  collate_fn=collate_batch, num_workers=0)
        by_length = sorted(range(len(valid_encoded)), key=lambda i: example_length(valid_encoded[i]))
        valid_batches = [by_length[i:i + batch_size] for i in range(0, len(by_length), batch_size)]
        valid_loader = DataLoader(valid_encoded, batch_sampler=valid_batches,
                                  collate_fn=collate_batch, num_workers=0)
        return train_loader, valid_loader
    train_loader = DataLoader(train_encoded, batch_size=batch_size, shuffle=True,
        collate_fn=collate_batch, num_workers=0,
        generator=torch.Generator().manual_seed(SEED))
    valid_loader = DataLoader(valid_encoded, batch_size=batch_size, shuffle=False,
        collate_fn=collate_batch, num_workers=0)
    return train_loader, valid_loader


# [IMPROVED] new helper | WHY: with bucketing the shuffling RNG lives in the batch sampler, not in the DataLoader
def loader_generator(loader):
    if isinstance(loader.batch_sampler, BucketBatchSampler):
        return loader.batch_sampler.generator
    return loader.generator


# [IMPROVED] new helper | WHY: measure how much of each target batch is padding (shows the effect of bucketing)
def padding_share(loader, max_batches=200):
    pad = total = 0
    for i, (_, tgt) in enumerate(loader):
        pad += int((tgt == 2).sum())
        total += tgt.numel()
        if i + 1 >= max_batches:
            break
    return pad / max(total, 1)

# Fingerprint exact encoded examples and their order to guard training resume.
def data_fingerprint():
    digest = hashlib.sha256()
    for name, pairs in [("train", train_encoded), ("validation", valid_encoded)]:
        digest.update(name.encode())
        for source, target in pairs:
            digest.update(json.dumps([source.tolist(), target.tolist()]).encode())
    return digest.hexdigest()

DATA_FINGERPRINT = data_fingerprint()
sample_src, sample_tgt = collate_batch(train_encoded[:4])
print("Batch shapes:", sample_src.shape, sample_tgt.shape)



> Batching matters a ton for speed. We want to have very evenly
> divided batches, with absolutely minimal padding. To do this we
> have to hack a bit around the default torchtext batching. This
> code patches their default batching to make sure we search over
> enough sentences to find tight batches.

## Iterators

In [ ]:
config = {"batch_size": 32, "num_epochs": 20, "accum_iter": 1,
          "base_lr": 1.0, "warmup": 1000}
model_settings = {"N": 2, "d_model": 128, "d_ff": 512, "h": 4, "dropout": 0.1}

# [IMPROVED] settings of the "improved" experiment ("baseline" keeps the original values above, unchanged)
if RUN_EXPERIMENT == "improved":
    config = {"batch_size": 64,    # WHY: was 32 - less noisy gradients and better GPU use
              "num_epochs": 20,
              "accum_iter": 1,
              "base_lr": 1.0,
              "warmup": 2000,      # WHY: was 1000 - peak LR = (d_model*warmup)^-0.5 = 1.4e-3 (baseline 2.8e-3), gentler for a bigger model
              "use_amp": True,     # WHY: float16 mixed precision -> faster training on the T4 GPU
              "clip_norm": 1.0,    # WHY: gradient clipping -> stable updates
              "bucketing": True}   # WHY: batches of similar length -> much less padding -> faster epochs
    model_settings = {"N": 3,             # WHY: was 2 - more layers = more capacity (the baseline was underfitting)
                      "d_model": 256,     # WHY: was 128
                      "d_ff": 1024,       # WHY: was 512 (kept at 4 x d_model as in the paper)
                      "h": 8,             # WHY: was 4 (d_k = 256 / 8 = 32 per head)
                      "dropout": 0.1,
                      "tie_weights": True}  # WHY: share target embedding and output projection (paper, section 3.4)

def atomic_save(payload, path):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)

def load_trained_model(path=None):
    path = BEST_MODEL_PATH if path is None else path
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    saved = torch.load(path, map_location="cpu", weights_only=True)
    en = Tokenizer.from_str(saved["tokenizer_en"])
    ar = Tokenizer.from_str(saved["tokenizer_ar"])
    sv, tv = vocabulary_from_tokenizer(en), vocabulary_from_tokenizer(ar)
    loaded = make_model(len(sv), len(tv), **saved["model_settings"]).to(device)
    loaded.load_state_dict(saved["model_state"])
    loaded.eval()
    return loaded, sv, tv, en, ar, saved["max_length"]

def train_model(resume=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Training on:", device)
    torch.manual_seed(SEED)
    model = make_model(len(vocab_src), len(vocab_tgt), **model_settings).to(device)
    print(f"Parameters: {count_parameters(model):,}")  # [IMPROVED] report model size
    criterion = LabelSmoothing(len(vocab_tgt), padding_idx=2, smoothing=0.1).to(device)
    loss_compute = SimpleLossCompute(model.generator, criterion)
    # [IMPROVED] was: create_dataloaders(config["batch_size"]) | WHY: pass the bucketing option (False for baseline)
    train_loader, valid_loader = create_dataloaders(config["batch_size"], config.get("bucketing", False))
    print(f"Batches per epoch: {len(train_loader):,} | "
          f"padding share of target positions: {padding_share(train_loader):.1%}")  # [IMPROVED] diagnostic
    loader_generator(train_loader).manual_seed(SEED)  # [IMPROVED] padding_share used one shuffle; reset -> same order as before
    optimizer = torch.optim.Adam(model.parameters(), lr=config["base_lr"],
                                 betas=(0.9, 0.98), eps=1e-9)
    scheduler = LambdaLR(optimizer, lr_lambda=lambda step: rate(
        step, model_settings["d_model"], 1.0, config["warmup"]))
    # [IMPROVED] mixed precision: autocast + GradScaler (disabled for baseline and on CPU -> original behaviour)
    use_amp = config.get("use_amp", False) and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print("Mixed precision (AMP):", use_amp, "| grad clip:", config.get("clip_norm"))
    state = TrainState()
    start_epoch, best_loss, history = 0, float("inf"), []

    if resume:
        saved = torch.load(LAST_MODEL_PATH, map_location="cpu", weights_only=True)
        if saved["data_fingerprint"] != DATA_FINGERPRINT:
            raise ValueError("Data changed: use a new run directory and train from scratch.")
        if saved["model_settings"] != model_settings:
            raise ValueError("Model settings differ from the saved run.")
        # [IMPROVED] also check the new options, with .get() so missing keys compare as None
        for key in ("batch_size", "accum_iter", "base_lr", "warmup",
                    "use_amp", "clip_norm", "bucketing"):
            if saved["config"].get(key) != config.get(key):
                raise ValueError(f"Resume requires unchanged {key}.")
        if (saved["tokenizer_en"] != tokenizer_en.to_str() or
            saved["tokenizer_ar"] != tokenizer_ar.to_str()):
            raise ValueError("Tokenizer mismatch.")
        model.load_state_dict(saved["model_state"])
        optimizer.load_state_dict(saved["optimizer_state"])
        scheduler.load_state_dict(saved["scheduler_state"])
        if saved.get("scaler_state"):  # [IMPROVED] restore the AMP loss scale when resuming
            scaler.load_state_dict(saved["scaler_state"])
        state = TrainState(**saved["train_state"])
        start_epoch, best_loss = saved["epoch"], saved["best_loss"]
        history = saved["history"]
        torch.set_rng_state(saved["torch_rng"])
        loader_generator(train_loader).set_state(saved["loader_rng"])  # [IMPROVED] was: train_loader.generator.set_state(...)
        if device.type == "cuda" and saved["cuda_rng"]:
            if len(saved["cuda_rng"]) == torch.cuda.device_count():
                torch.cuda.set_rng_state_all(saved["cuda_rng"])
        if not BEST_MODEL_PATH.exists():
            raise FileNotFoundError("Keep best.pt and last.pt together when resuming.")
        print("Resuming after epoch", start_epoch)

    def batches(loader):
        for src, tgt in loader:
            yield Batch(src.to(device), tgt.to(device), pad=2)

    epoch_times = []  # [IMPROVED] measured epoch durations -> ETA
    for epoch in range(start_epoch, config["num_epochs"]):
        print(f"\nEpoch {epoch+1}/{config['num_epochs']}")
        epoch_start = time.time()  # [IMPROVED]
        # [IMPROVED] pass scaler / clip_norm / use_amp (all off for baseline). log_every=500
        # WHY: the old run printed ~80 lines per epoch; 500 keeps the output readable.
        train_loss, state = run_epoch(batches(train_loader), model, loss_compute,
            optimizer, scheduler, mode="train", accum_iter=config["accum_iter"],
            train_state=state, scaler=scaler, clip_norm=config.get("clip_norm"),
            use_amp=use_amp, log_every=500)
        val_loss, _ = run_epoch(batches(valid_loader), model, loss_compute,
            DummyOptimizer(), DummyScheduler(), mode="eval", use_amp=use_amp)
        if not math.isfinite(train_loss) or not math.isfinite(val_loss):
            raise RuntimeError("Non-finite loss: checkpoints were not overwritten.")
        seconds = time.time() - epoch_start  # [IMPROVED]
        epoch_times.append(seconds)
        history.append({"epoch": epoch+1, "train_loss": train_loss,
                        "validation_loss": val_loss,
                        "epoch_seconds": seconds})  # [IMPROVED] epoch time kept in the history
        # [IMPROVED] also print perplexity = exp(loss), the epoch time and the estimated remaining time
        remaining = (config["num_epochs"] - epoch - 1) * sum(epoch_times) / len(epoch_times)
        print(f"Train: {train_loss:.4f} | Validation: {val_loss:.4f} | "
              f"PPL(val): {math.exp(val_loss):.1f} | epoch: {seconds / 60:.1f} min | "
              f"ETA: {remaining / 60:.0f} min"
              + (f" | AMP skipped steps: {state.skipped_steps}" if use_amp else ""))
        improved = val_loss < best_loss
        best_loss = min(best_loss, val_loss)
        payload = {
            "model_state": model.state_dict(), "model_settings": model_settings,
            "tokenizer_en": tokenizer_en.to_str(), "tokenizer_ar": tokenizer_ar.to_str(),
            "max_length": MAX_LENGTH, "epoch": epoch+1, "best_loss": best_loss,
            "validation_loss": val_loss, "config": dict(config), "history": history,
            "data_fingerprint": DATA_FINGERPRINT,
            # [IMPROVED] extra information for the evaluation table
            "experiment": RUN_EXPERIMENT, "num_parameters": count_parameters(model),
            "train_pairs": len(train_encoded),
        }
        if improved:
            atomic_save(payload, BEST_MODEL_PATH)
            print("Saved best validation model.")
        payload.update({
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(), "train_state": vars(state),
            "scaler_state": scaler.state_dict() if use_amp else {},  # [IMPROVED] needed to resume AMP training
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
            "loader_rng": loader_generator(train_loader).get_state(),  # [IMPROVED] was: train_loader.generator.get_state()
        })
        atomic_save(payload, LAST_MODEL_PATH)
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device,
                                     weights_only=True)["model_state"])
    model.eval()
    return model, pd.DataFrame(history)


## Training the System

In [ ]:
if RUN_MODE == "load":
    model, vocab_src, vocab_tgt, tokenizer_en, tokenizer_ar, MAX_LENGTH = load_trained_model()
    training_history = pd.DataFrame(torch.load(
        BEST_MODEL_PATH, map_location="cpu", weights_only=True)["history"])
else:
    model, training_history = train_model(resume=(RUN_MODE == "resume"))
display(training_history)


Training on: cuda

Epoch 1/20
Batch:     0 | Loss/token: 7.7722 | Tokens/sec: 542.1 | LR: 2.80e-06
Batch:    40 | Loss/token: 7.5406 | Tokens/sec: 10090.3 | LR: 1.15e-04
Batch:    80 | Loss/token: 7.0230 | Tokens/sec: 12554.9 | LR: 2.26e-04
Batch:   120 | Loss/token: 6.5952 | Tokens/sec: 13912.4 | LR: 3.38e-04
Batch:   160 | Loss/token: 6.4766 | Tokens/sec: 14672.7 | LR: 4.50e-04
Batch:   200 | Loss/token: 6.4071 | Tokens/sec: 15124.0 | LR: 5.62e-04
Batch:   240 | Loss/token: 6.2756 | Tokens/sec: 15449.8 | LR: 6.74e-04
Batch:   280 | Loss/token: 6.2861 | Tokens/sec: 15573.7 | LR: 7.85e-04
Batch:   320 | Loss/token: 6.2034 | Tokens/sec: 15642.4 | LR: 8.97e-04
Batch:   360 | Loss/token: 5.7819 | Tokens/sec: 15832.2 | LR: 1.01e-03
Batch:   400 | Loss/token: 5.9630 | Tokens/sec: 15960.2 | LR: 1.12e-03
Batch:   440 | Loss/token: 5.8645 | Tokens/sec: 16037.9 | LR: 1.23e-03
Batch:   480 | Loss/token: 5.5623 | Tokens/sec: 16110.1 | LR: 1.34e-03
Batch:   520 | Loss/token: 5.7359 | Tokens/sec: 1

,epoch,train_loss,validation_loss
0,1,5.313832,4.650300
1,2,4.470129,4.302966
2,3,4.205755,4.145915
3,4,4.055878,4.054391
4,5,3.954580,3.998669
5,6,3.880601,3.930880
6,7,3.818662,3.892499
7,8,3.768627,3.854773
8,9,3.725869,3.834349
9,10,3.687860,3.801263


In [ ]:
# Optional later reload; the checkpoint includes both exact BPE tokenizers.
# model, vocab_src, vocab_tgt, tokenizer_en, tokenizer_ar, MAX_LENGTH = load_trained_model()



> Once trained we can decode the model to produce a set of
> translations. Here we simply translate the first sentence in the
> validation set. This dataset is pretty small so the translations
> with greedy search are reasonably accurate.

> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.

# Results

On the WMT 2014 English-to-German translation task, the big
transformer model (Transformer (big) in Table 2) outperforms the
best previously reported models (including ensembles) by more than
2.0 BLEU, establishing a new state-of-the-art BLEU score of
28.4. The configuration of this model is listed in the bottom line
of Table 3. Training took 3.5 days on 8 P100 GPUs. Even our base
model surpasses all previously published models and ensembles, at a
fraction of the training cost of any of the competitive models.

On the WMT 2014 English-to-French translation task, our big model
achieves a BLEU score of 41.0, outperforming all of the previously
published single models, at less than 1/4 the training cost of the
previous state-of-the-art model. The Transformer (big) model trained
for English-to-French used dropout rate Pdrop = 0.1, instead of 0.3.


![](https://github.com/harvardnlp/annotated-transformer/blob/master/images/results.png?raw=1)

> **Original paper context:** the text below describes the paper, not measured results of this English–Arabic run. This notebook uses a small Pre-LayerNorm model with separate embeddings.



> With the addtional extensions in the last section, the OpenNMT-py
> replication gets to 26.9 on EN-DE WMT. Here I have loaded in those
> parameters to our reimplemenation.


In [ ]:
# Load data and model for output checks

In [ ]:
def decode_bpe(ids, tokenizer):
    content = []
    for token_id in ids:
        token_id = int(token_id)
        if token_id == 1:
            break
        if token_id not in (0, 2):
            content.append(token_id)
    return tokenizer.decode(content, skip_special_tokens=False)

# [IMPROVED] new helper | WHY: a translation rarely needs more than ~2x the source length; stops runaway outputs early
def output_length_cap(source_length, max_length=None):
    max_length = MAX_LENGTH if max_length is None else max_length
    return min(max_length, 2 * source_length + 10)

# [IMPROVED] was: translate(text) with the global model and greedy decoding only.
# WHY: the test-set evaluation needs to translate with ANY saved model (baseline / improved) and with beam search.
#      Called as translate(text) it still uses the global model, tokenizers and greedy decoding.
@torch.no_grad()
def translate(text, net=None, tok_src=None, tok_tgt=None, max_length=None,
              beam_size=1, length_penalty=0.6):
    net = model if net is None else net
    tok_src = tokenizer_en if tok_src is None else tok_src
    tok_tgt = tokenizer_ar if tok_tgt is None else tok_tgt
    max_length = MAX_LENGTH if max_length is None else max_length
    device = next(net.parameters()).device
    # [IMPROVED] was: raise ValueError for long inputs | WHY: truncate instead, so no test sentence crashes the evaluation
    ids = tok_src.encode(text).ids[: max_length - 2]
    src = torch.tensor([[0] + ids + [1]], dtype=torch.long, device=device)
    src_mask = (src != 2).unsqueeze(-2)
    cap = output_length_cap(src.size(1), max_length)
    if beam_size > 1:
        out = beam_search_decode(net, src, src_mask, cap, 0, 1,
                                 beam_size=beam_size, length_penalty=length_penalty)
    else:
        out = greedy_decode(net, src, src_mask, cap, 0, 1)
    return decode_bpe(out[0].tolist(), tok_tgt)

@torch.no_grad()
def run_model_example(n_examples=5):
    model.eval()
    device = next(model.parameters()).device
    results = []
    for i, (src_ids, tgt_ids) in enumerate(valid_encoded[:n_examples]):
        src = src_ids.unsqueeze(0).to(device)
        out = greedy_decode(model, src, (src != 2).unsqueeze(-2), MAX_LENGTH, 0, 1)
        en, ar = valid_texts_kept[i]
        prediction = decode_bpe(out[0].tolist(), tokenizer_ar)
        print(f"\nExample {i+1}\nEnglish: {en}\nArabic reference: {ar}\nPrediction: {prediction}")
        print(f"Beam-4:     {translate(en, beam_size=4)}")  # [IMPROVED] beam-search output next to the greedy one
        results.append({"src": src.cpu(), "target": tgt_ids.unsqueeze(0).cpu(),
            "output": out.cpu(), "source_text": en, "reference_text": ar,
            "prediction_text": prediction})
    return model, results

model, example_data = run_model_example()
# Custom example: print(translate("Where is the vault?"))


In [ ]:
# [IMPROVED] NEW CELL: test-set evaluation with BLEU and chrF
#
# WHY: validation loss chose the best epoch, so it is no longer an unbiased measure, and per-token losses from
#      different BPE tokenizers are not comparable. The OPUS-100 test split was never used for training,
#      tokenizers or model selection, and every experiment is scored on the SAME test sentences.
#
# BLEU (word n-grams, n = 1..4):
#   p_n  = clipped matching n-grams / n-grams in the hypothesis   (clipping: "the the the" cannot match 3 times)
#   BLEU = BP * exp( sum_n 1/4 * log p_n ),   BP = 1 if c > r else exp(1 - r/c)   (c = hyp length, r = ref length)
# chrF (character n-grams, n = 1..6):
#   chrF = (1 + b^2) * P * R / (b^2 * P + R),  b = 2 (recall counts twice as much as precision)
#   Better suited to Arabic: clitics (و / ال / ها) are attached to words, so a small spelling difference
#   gives 0 for that word in BLEU but still partial credit in chrF.
import sacrebleu

EVAL_LIMIT = 1000   # test sentences scored per experiment
BEAM_SIZE = 4
LENGTH_PENALTY = 0.6

# Fixed test set: same seed and filters for every experiment (no tokenizer-based filtering).
test_texts = collect_text_pairs(raw_data["test"], EVAL_LIMIT)
test_sources = [en for en, _ in test_texts]
test_references = [ar for _, ar in test_texts]
train_sources = {en for en, _ in train_texts}
overlap = sum(en in train_sources for en in test_sources)
print(f"Test sentences: {len(test_texts):,} | source also seen in this run's training data: "
      f"{overlap} ({overlap / max(len(test_texts), 1):.1%})")
pd.DataFrame(test_texts, columns=["english", "arabic_reference"]).to_csv(
    BASE_DIR / "test_set_used.csv", index=False)


@torch.no_grad()
def translate_corpus(net, tok_src, tok_tgt, max_length, sources,
                     beam_size=1, batch_size=64, length_penalty=LENGTH_PENALTY):
    """Translate a list of sentences. Greedy is batched; beam search runs per sentence."""
    if beam_size > 1:
        return [translate(text, net, tok_src, tok_tgt, max_length, beam_size, length_penalty)
                for text in sources]
    device = next(net.parameters()).device
    encoded = [torch.tensor([0] + tok_src.encode(t).ids[: max_length - 2] + [1]) for t in sources]
    order = sorted(range(len(encoded)), key=lambda i: encoded[i].numel())
    outputs = [None] * len(encoded)
    for start in range(0, len(order), batch_size):
        chunk = order[start:start + batch_size]
        src = pad_sequence([encoded[i] for i in chunk], batch_first=True,
                           padding_value=2).to(device)
        cap = output_length_cap(src.size(1), max_length)
        out = greedy_decode_batch(net, src, (src != 2).unsqueeze(-2), cap, 0, 1, 2)
        for row, index in enumerate(chunk):
            own_cap = output_length_cap(encoded[index].numel(), max_length)
            outputs[index] = decode_bpe(out[row, :own_cap].tolist(), tok_tgt)
    return outputs


def score_translations(hypotheses, references):
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])
    return {"BLEU": round(bleu.score, 2), "chrF": round(chrf.score, 2),
            "length_ratio": round(bleu.sys_len / max(bleu.ref_len, 1), 3)}

In [ ]:
# [IMPROVED] NEW CELL: score every trained experiment (baseline / improved) with greedy and beam search
# Run RUN_EXPERIMENT = "baseline" first, then "improved"; this cell finds both checkpoints and compares them.
#   - baseline (greedy) vs improved (greedy) -> effect of the model / data / training changes
#   - greedy vs beam-4 inside one experiment -> effect of beam search alone
#   - length_ratio near 1.0 = outputs as long as the references (below ~0.9 the BLEU brevity penalty hurts)
evaluation_rows, sample_rows, histories = [], {}, {}

for name in ("baseline", "improved"):
    path = BASE_DIR / name / "best.pt"
    if not path.exists():
        print(f"[{name}] no checkpoint at {path} - train it first (RUN_EXPERIMENT = '{name}').")
        continue
    net, _, _, tok_src, tok_tgt, max_len = load_trained_model(path)
    info = torch.load(path, map_location="cpu", weights_only=True)
    histories[name] = pd.DataFrame(info["history"])
    sample_rows.setdefault("english", test_sources[:10])
    sample_rows.setdefault("reference", test_references[:10])
    for beam in (1, BEAM_SIZE):
        decoder_name = "greedy" if beam == 1 else f"beam-{beam}"
        start = time.time()
        hypotheses = translate_corpus(net, tok_src, tok_tgt, max_len, test_sources, beam_size=beam)
        seconds = time.time() - start
        row = {"experiment": name, "decoding": decoder_name,
               "parameters": count_parameters(net), "train_pairs": info.get("train_pairs"),
               "best_epoch": info["epoch"], "best_val_loss": round(info["best_loss"], 4),
               **score_translations(hypotheses, test_references),
               "decode_seconds": round(seconds, 1)}
        evaluation_rows.append(row)
        sample_rows[f"{name} ({decoder_name})"] = hypotheses[:10]
        print(row)
        pd.DataFrame({"english": test_sources, "reference": test_references,
                      "hypothesis": hypotheses}).to_csv(
            BASE_DIR / f"test_translations_{name}_{decoder_name}.csv", index=False)
    del net
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_table = pd.DataFrame(evaluation_rows)
if len(results_table):
    results_table.to_csv(BASE_DIR / "test_results.csv", index=False)
    display(results_table)
    display(pd.DataFrame(sample_rows))

In [ ]:
# [IMPROVED] NEW CELL: validation curves of both experiments on one chart.
# Note: each run has its own BPE tokenizers, so loss per token is only roughly comparable;
# the BLEU / chrF table above is the fair comparison.
if histories:
    curves = pd.concat([h.assign(experiment=n) for n, h in histories.items()])
    display(
        alt.Chart(curves)
        .mark_line(point=True)
        .encode(
            x=alt.X("epoch:Q", title="Epoch"),
            y=alt.Y("validation_loss:Q", title="Validation loss per BPE token",
                    scale=alt.Scale(zero=False)),
            color="experiment:N",
            tooltip=["experiment:N", "epoch:Q", "validation_loss:Q"],
        )
        .properties(width=650, height=300)
    )

## Attention visualization

Refresh attention on the chosen example before plotting. Arabic is decoded normally in translations. For heatmap labels, a subword is decoded alone when possible; a partial UTF-8 character is labeled with its token ID and raw byte token instead. These are subword positions, not word alignments. Decoder query labels refer to input positions that predict the next token. Plots cap displayed dimensions only, not the actual attention computation.


In [ ]:
def attention_labels(ids, tokenizer):
    labels = []
    for token_id in ids:
        token_id = int(token_id)
        raw = tokenizer.id_to_token(token_id)
        decoded = tokenizer.decode([token_id], skip_special_tokens=False)
        if "\ufffd" in decoded:
            decoded = f"byte:{token_id} {raw}"
        labels.append(decoded.replace(" ", "▁").replace("\n", "↵"))
    return labels

@torch.no_grad()
def prepare_attention(model, example, use_reference=False):
    model.eval()
    device = next(model.parameters()).device

    src = example["src"].to(device)
    full_target = (
        example["target"] if use_reference else example["output"]
    ).to(device)

    # Match next-token prediction: omit the final output token.
    decoder_input = full_target[:, :-1]

    if decoder_input.size(1) == 0:
        raise ValueError("The selected output has no decoder input tokens.")

    src_mask = (src != vocab_src["<blank>"]).unsqueeze(-2)
    tgt_mask = Batch.make_std_mask(
        decoder_input, vocab_tgt["<blank>"]
    )

    # Refresh all attention matrices for THIS example.
    model(src, decoder_input, src_mask, tgt_mask)

    source_labels = attention_labels(src[0].tolist(), tokenizer_en)
    decoder_labels = attention_labels(decoder_input[0].tolist(), tokenizer_ar)

    return source_labels, decoder_labels


def get_encoder(model, layer):
    return model.encoder.layers[layer].self_attn.attn


def get_decoder_self(model, layer):
    return model.decoder.layers[layer].self_attn.attn


def get_decoder_src(model, layer):
    return model.decoder.layers[layer].src_attn.attn


In [ ]:
def mtx2df(matrix, row_tokens, col_tokens, max_dim=30):
    matrix = matrix.detach().cpu()

    n_rows = min(matrix.size(0), len(row_tokens), max_dim)
    n_cols = min(matrix.size(1), len(col_tokens), max_dim)

    return pd.DataFrame([
        {
            "row": r,
            "column": c,
            "value": matrix[r, c].item(),
            "row_token": f"{r:03d} {row_tokens[r]}",
            "col_token": f"{c:03d} {col_tokens[c]}",
        }
        for r in range(n_rows)
        for c in range(n_cols)
    ])


def attn_map(
    attn, layer, head, row_tokens, col_tokens, max_dim=30
):
    df = mtx2df(
        attn[0, head], row_tokens, col_tokens, max_dim
    )

    return (
        alt.Chart(df)
        .mark_rect()
        .encode(
            x=alt.X(
                "col_token:O",
                sort=None,
                title="Key tokens",
                axis=alt.Axis(labelLimit=150),
            ),
            y=alt.Y(
                "row_token:O",
                sort=None,
                title="Query tokens",
                axis=alt.Axis(labelLimit=150),
            ),
            color=alt.Color(
                "value:Q",
                title="Attention",
                scale=alt.Scale(domain=[0, 1], scheme="viridis"),
            ),
            tooltip=[
                "row_token:N",
                "col_token:N",
                alt.Tooltip("value:Q", format=".4f"),
            ],
        )
        .properties(
            width=300,
            height=300,
            title=f"Layer {layer + 1}, Head {head + 1}",
        )
    )


def visualize_layer(
    model,
    layer,
    getter_fn,
    row_tokens,
    col_tokens,
    max_dim=30,
):
    attn = getter_fn(model, layer)

    if attn is None:
        raise ValueError("Run prepare_attention() first.")

    charts = [
        attn_map(
            attn,
            layer,
            head,
            row_tokens,
            col_tokens,
            max_dim=max_dim,
        )
        for head in range(attn.size(1))
    ]

    rows = [
        alt.hconcat(*charts[i:i + 2])
        for i in range(0, len(charts), 2)
    ]

    return alt.vconcat(*rows)

In [ ]:
source_labels, decoder_labels = prepare_attention(
    model,
    example_data[0],
    use_reference=False,
)

In [ ]:
display(visualize_layer(
    model,
    layer=0,
    getter_fn=get_encoder,
    row_tokens=source_labels,
    col_tokens=source_labels,
))

alt.VConcatChart(...)

In [ ]:
display(visualize_layer(
    model,
    layer=0,
    getter_fn=get_decoder_self,
    row_tokens=decoder_labels,
    col_tokens=decoder_labels,
))

alt.VConcatChart(...)

In [ ]:
# The cross-attention chart follows in the next cell.


In [ ]:
display(visualize_layer(
    model,
    layer=0,
    getter_fn=get_decoder_src,
    row_tokens=decoder_labels,
    col_tokens=source_labels,
))

alt.VConcatChart(...)

## Encoder Self Attention

In [ ]:
def visualize_attention_stack(
    getter_fn,
    attention_type,
    example_index=0,
    use_reference=False,
    max_dim=30,
):
    example = example_data[example_index]

    # Refresh attention for the selected example.
    source_labels, decoder_labels = prepare_attention(
        model,
        example,
        use_reference=use_reference,
    )

    if attention_type == "encoder":
        row_tokens = source_labels
        col_tokens = source_labels
        n_layers = len(model.encoder.layers)
        title = "Encoder self-attention: English → English"

    elif attention_type == "decoder":
        row_tokens = decoder_labels
        col_tokens = decoder_labels
        n_layers = len(model.decoder.layers)
        title = "Decoder masked self-attention: Arabic → Arabic"

    elif attention_type == "cross":
        row_tokens = decoder_labels
        col_tokens = source_labels
        n_layers = len(model.decoder.layers)
        title = "Cross-attention: Arabic queries → English keys"

    else:
        raise ValueError("Unknown attention type.")

    layer_charts = [
        visualize_layer(
            model=model,
            layer=layer,
            getter_fn=getter_fn,
            row_tokens=row_tokens,
            col_tokens=col_tokens,
            max_dim=max_dim,
        )
        for layer in range(n_layers)
    ]

    return alt.vconcat(*layer_charts).properties(title=title)


def viz_encoder_self(example_index=0, max_dim=30):
    return visualize_attention_stack(
        get_encoder,
        "encoder",
        example_index=example_index,
        max_dim=max_dim,
    )


def viz_decoder_self(
    example_index=0, use_reference=False, max_dim=30
):
    return visualize_attention_stack(
        get_decoder_self,
        "decoder",
        example_index=example_index,
        use_reference=use_reference,
        max_dim=max_dim,
    )


def viz_decoder_src(
    example_index=0, use_reference=False, max_dim=30
):
    return visualize_attention_stack(
        get_decoder_src,
        "cross",
        example_index=example_index,
        use_reference=use_reference,
        max_dim=max_dim,
    )

In [ ]:
show_example(viz_encoder_self)

alt.VConcatChart(...)

In [ ]:
show_example(viz_decoder_self)

alt.VConcatChart(...)

In [ ]:
show_example(viz_decoder_src)

alt.VConcatChart(...)

In [ ]:
display(viz_decoder_src(
    example_index=1,
    use_reference=True,
))

alt.VConcatChart(...)

# Conclusion

 Hopefully this code is useful for future research. Please reach
 out if you have any issues.


 Cheers,
 Sasha Rush, Austin Huang, Suraj Subramanian, Jonathan Sum, Khalid Almubarak,
 Stella Biderman

In [ ]:
# Coverage diagnostic for the BPE data (not the old word vocabulary).
for split_name, pairs in [("Training", train_encoded), ("Validation", valid_encoded)]:
    print(split_name,
          f"English unknown: {encoded_unknown_rate(pairs, 0):.2%}",
          f"Arabic unknown: {encoded_unknown_rate(pairs, 1):.2%}")
print("Original validation reference:", valid_texts_kept[0][1])


Training English unknown: 0.00% Arabic unknown: 0.00%
Validation English unknown: 0.00% Arabic unknown: 0.00%
Original validation reference: أين الكنز الثمين؟


In [ ]:
display(training_history)

best_row = training_history.loc[
    training_history["validation_loss"].idxmin()
]
print("Best validation epoch:", int(best_row["epoch"]))

display(
    alt.Chart(
        training_history.melt(
            id_vars="epoch",
            value_vars=["train_loss", "validation_loss"],
            var_name="Split",
            value_name="Loss",
        )
    )
    .mark_line(point=True)
    .encode(
        x=alt.X("epoch:Q", title="Epoch"),
        y=alt.Y("Loss:Q", title="Loss per BPE token"),
        color="Split:N",
        tooltip=["epoch:Q", "Split:N", "Loss:Q"],
    )
    .properties(width=650, height=300)
)

,epoch,train_loss,validation_loss
0,1,5.313832,4.650300
1,2,4.470129,4.302966
2,3,4.205755,4.145915
3,4,4.055878,4.054391
4,5,3.954580,3.998669
5,6,3.880601,3.930880
6,7,3.818662,3.892499
7,8,3.768627,3.854773
8,9,3.725869,3.834349
9,10,3.687860,3.801263


Best validation epoch: 20


alt.Chart(...)